In [ ]:
!pip install pyngrok flask

In [ ]:
# Load your model and tokenizer here
# import torch
# from transformers import AutoModelForCausalLM, AutoTokenizer
# 
# model_name = "mistralai/Mistral-7B-Instruct-v0.3"
# tokenizer = AutoTokenizer.from_pretrained(model_name)
# model = AutoModelForCausalLM.from_pretrained(model_name, torch_dtype=torch.float16, device_map="auto")

In [ ]:
import threading
from flask import Flask, request, jsonify
from pyngrok import ngrok

# 1. Set your ngrok auth token (Get it from https://dashboard.ngrok.com/get-started/your-authtoken)
# Uncomment the line below and replace with your token
# ngrok.set_auth_token("YOUR_NGROK_AUTH_TOKEN")

app = Flask(__name__)

def generate_text(prompt, max_length=700, num_return_sequences=1):
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    outputs = model.generate(
        **inputs,
        max_new_tokens=max_length,
        num_return_sequences=num_return_sequences,
        do_sample=True,
        top_k=50,
        top_p=0.95,
        temperature=0.7,
        pad_token_id=tokenizer.eos_token_id,
        repetition_penalty=1.3,
        no_repeat_ngram_size=4,
    )
    input_len = inputs["input_ids"].shape[1]
    return [tokenizer.decode(o[input_len:], skip_special_tokens=True) for o in outputs]

def messages_to_prompt(messages: list) -> str:
    system = ""
    user = ""
    for m in messages:
        if m["role"] == "system":
            system = m["content"]
        elif m["role"] == "user":
            user = m["content"]
    combined = f"{system}\n\n{user}" if system else user
    return f"<s>[INST] {combined} [/INST]"

@app.route("/v1/chat/completions", methods=["POST"])
def chat_completions():
    data = request.get_json()
    messages = data.get("messages", [])
    max_tokens = data.get("max_tokens", 700)

    prompt = messages_to_prompt(messages)
    text = generate_text(prompt, max_length=max_tokens, num_return_sequences=1)[0]

    return jsonify({
        "choices": [
            {"message": {"role": "assistant", "content": text}}
        ]
    })

def run_server():
    # Run Flask with werkzeug logging disabled to keep the notebook output clean
    import logging
    log = logging.getLogger('werkzeug')
    log.setLevel(logging.ERROR)
    app.run(host="0.0.0.0", port=8000)

# 2. Open a ngrok tunnel to the HTTP server
public_url = ngrok.connect(8000).public_url
print(f"[*] Ngrok Tunnel URL: {public_url}/v1/chat/completions")
print("[*] Copy the URL above and use it in your local terminal as the --api-url argument.")

# 3. Start the Flask server in a background thread
server_thread = threading.Thread(target=run_server, daemon=True)
server_thread.start()

print("[*] API Server is running in the background.")